# Sprint 1: Infraestructura de Datos y Pipeline de Calibracion

**Objetivo:** Consolidar el entorno de datos local en WSL2 y construir el cargador de tensores
que alimentara los algoritmos de cuantizacion offline (GPTQ y AWQ) en la RTX 4060 Ti.

**Dataset:** [fsicoli/common_voice_17_0](https://huggingface.co/datasets/fsicoli/common_voice_17_0) (config `es`)
- Mirror comunitario completo de Mozilla Common Voice 17.0 Spanish (~2,220h total, 562h validadas)
- Splits nativos: `train`, `validation`, `test`, `other`, `invalidated`
- Audio original a 16kHz mono, licencia CC0-1.0
- **Motivo del cambio:** Mozilla retiro CV 17.0 de HuggingFace en Oct 2025. El mirror `fsicoli` almacena los audios directamente.

> **Nota tecnica:** VibeVoice 1.5B procesa audio crudo a **24kHz mono**, no espectrogramas de Mel.
> El preprocesamiento consiste en remuestreo + normalizacion de amplitud, no en conversion frecuencial.

---
## Bloque 0: Configuracion del Entorno y Parches Criticos

Se aplica el parche obligatorio de `transformers` para evitar colisiones de mapeo
con el fork comunitario de VibeVoice (ver `agents.md` seccion 3).

In [3]:
import sys
import os
import json
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import transformers
from datasets import load_dataset, Audio, Dataset, DatasetDict, concatenate_datasets
from tqdm.auto import tqdm

# --- Parche critico: colision de mapeo en transformers ---
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)
print("Parche de CONFIG_MAPPING aplicado.")

# --- Rutas del proyecto ---
PROJECT_ROOT = Path(os.path.abspath("")).parent if "notebooks" in os.path.abspath("") else Path(os.path.abspath(""))
if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"WEIGHTS_DIR  : {WEIGHTS_DIR}")
print(f"Torch        : {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Parche de CONFIG_MAPPING aplicado.
PROJECT_ROOT : /home/alfrog/projects/VibeVoice_Optimization
DATA_DIR     : /home/alfrog/projects/VibeVoice_Optimization/data
WEIGHTS_DIR  : /home/alfrog/projects/VibeVoice_Optimization/weights
Torch        : 2.12.0+cu130  |  CUDA: True
GPU          : NVIDIA GeForce RTX 4060 Ti
VRAM Total   : 8.00 GB


---
## TSK-1.1: Descarga Automatizada del Dataset

Se descarga el subset en espanol de Common Voice 17.0 desde el mirror comunitario
`fsicoli/common_voice_17_0` con config `"es"`. El dataset contiene las particiones
nativas `train`/`validation`/`test`/`other` con audios y transcripciones.

**Estrategia:** Cargamos el split `train` como base para el 70% de datos de
entrenamiento (fine-tuning), y los splits `test` y `validation` nativos como
nuestros sets de benchmark y calibracion respectivamente.

In [4]:
DATASET_ID = "fsicoli/common_voice_17_0"
DATASET_CONFIG = "es"

print(f"Descargando dataset: {DATASET_ID} (config={DATASET_CONFIG})")
print("(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)\n")

t_start = time.time()

cv17_dataset = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split="train",
    trust_remote_code=True,
)

t_elapsed = time.time() - t_start
print(f"\nDescarga completada en {t_elapsed/60:.1f} minutos.")
print(f"Registros cargados (train): {len(cv17_dataset):,}")
print(f"Columnas: {cv17_dataset.column_names}")

Descargando dataset: fsicoli/common_voice_17_0 (config=es)
(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)



Reading metadata...: 336846it [00:01, 302211.76it/s]s/s]
Generating train split: 336846 examples [00:40, 8267.73 examples/s] 
Reading metadata...: 15857it [00:00, 328621.65it/s]xamples/s]
Generating validation split: 15857 examples [00:02, 7788.17 examples/s]
Reading metadata...: 15857it [00:00, 350204.45it/s]s/s]
Generating test split: 15857 examples [00:02, 7865.53 examples/s]
Reading metadata...: 1138631it [00:03, 311900.66it/s]/s]
Generating other split: 1138631 examples [02:04, 9145.43 examples/s] 
Reading metadata...: 86227it [00:00, 352991.03it/s]examples/s]
Generating invalidated split: 86227 examples [00:09, 8963.00 examples/s] 



Descarga completada en 28.6 minutos.
Registros cargados (train): 336,846
Columnas: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant']


---
## Exploracion del Dataset

Inspeccionamos la estructura, metadatos demograficos y distribucion de duraciones
para validar la calidad de la muestra antes de la particion.

In [5]:
# --- Muestra de registros ---
print("=== Ejemplos del dataset ===\n")
for i in range(3):
    sample = cv17_dataset[i]
    audio_info = sample["audio"]
    duration_s = len(audio_info["array"]) / audio_info["sampling_rate"]
    print(f"[{i}] \"{sample['sentence'][:80]}...\"")
    print(f"    Duracion: {duration_s:.2f}s | SR: {audio_info['sampling_rate']}Hz")
    print(f"    Locale: {sample['locale']} | Genero: {sample['gender']} | Edad: {sample['age']}")
    print()

# --- Distribucion de duraciones (muestreo estadistico) ---
SAMPLE_SIZE = 5000
indices = random.sample(range(len(cv17_dataset)), min(SAMPLE_SIZE, len(cv17_dataset)))
durations = []
for idx in tqdm(indices, desc="Muestreando duraciones"):
    arr = cv17_dataset[idx]["audio"]["array"]
    sr = cv17_dataset[idx]["audio"]["sampling_rate"]
    durations.append(len(arr) / sr)

durations = np.array(durations)
print(f"\n=== Estadisticas de Duracion (muestra n={len(durations)}) ===")
print(f"Media    : {durations.mean():.2f}s")
print(f"Mediana  : {np.median(durations):.2f}s")
print(f"Min/Max  : {durations.min():.2f}s / {durations.max():.2f}s")
print(f"Std      : {durations.std():.2f}s")
print(f"Horas est. totales: {durations.mean() * len(cv17_dataset) / 3600:.1f}h")

# --- Distribucion de genero y edad ---
genders = Counter()
ages = Counter()
for idx in tqdm(indices, desc="Muestreando demografia"):
    sample = cv17_dataset[idx]
    genders[sample["gender"]] += 1
    ages[sample["age"]] += 1

print(f"\nGeneros  : {dict(genders.most_common())}")
print(f"Edades    : {dict(ages.most_common())}")

=== Ejemplos del dataset ===

[0] "A ese grupo de amigos les gusta comer helados por la tarde...."
    Duracion: 3.89s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[1] "A mi profesora le importa demasiado la gramática y muy poco el contenido...."
    Duracion: 5.16s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[2] "A decir verdad , eres demasiado genial ...."
    Duracion: 3.05s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties



Muestreando duraciones: 100%|██████████| 5000/5000 [00:33<00:00, 148.61it/s]



=== Estadisticas de Duracion (muestra n=5000) ===
Media    : 5.19s
Mediana  : 5.11s
Min/Max  : 1.58s / 10.44s
Std      : 1.44s
Horas est. totales: 485.2h


Muestreando demografia: 100%|██████████| 5000/5000 [00:13<00:00, 360.70it/s]


Generos  : {'male_masculine': 2864, 'female_feminine': 1576, '': 560}
Edades    : {'twenties': 2387, 'thirties': 616, '': 542, 'sixties': 509, 'fifties': 389, 'teens': 292, 'fourties': 262, 'seventies': 1, 'nineties': 1, 'eighties': 1}


---
## TSK-1.2: Particion de la Muestra

Common Voice 17.0 ya proporciona divisiones nativas. Las usamos directamente
para mantener trazabilidad con el estandar de la comunidad:

| Particion | Split CV17 | Proposito en el proyecto |
|-----------|-----------|---------------------------|
| Test      | `test`    | Benchmark final (Sprint 5) |
| Validacion| `validation` + muestra `other` | Calibracion offline (GPTQ/AWQ) |
| Train     | `train`   | Fine-tuning espanol (Sprint 1.5) |

Cargamos todos los splits y guardamos los indices para trazabilidad.

In [6]:
# --- Cargar todos los splits nativos de CV17 es ---
print("Cargando splits nativos de Common Voice 17.0 (es)...")

cv17_all = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    trust_remote_code=True,
)

for split_name, split_data in cv17_all.items():
    print(f"  {split_name:<20} {len(split_data):>10,} registros")

train_dataset = cv17_all["train"]
test_dataset = cv17_all["test"]
val_dataset = cv17_all["validation"]
other_dataset = cv17_all.get("other", None)

print(f"\n=== Asignacion para el proyecto ===")
print(f"Train (fine-tuning)       : {len(train_dataset):,} registros")
print(f"Test (benchmark)          : {len(test_dataset):,} registros")
print(f"Validation (calibracion)  : {len(val_dataset):,} registros")
if other_dataset:
    print(f"Other (calibracion extra) : {len(other_dataset):,} registros")

Cargando splits nativos de Common Voice 17.0 (es)...
  train                   336,846 registros
  validation               15,857 registros
  test                     15,857 registros
  other                 1,138,631 registros
  invalidated              86,227 registros

=== Asignacion para el proyecto ===
Train (fine-tuning)       : 336,846 registros
Test (benchmark)          : 15,857 registros
Validation (calibracion)  : 15,857 registros
Other (calibracion extra) : 1,138,631 registros


In [7]:
# --- Guardar metadatos de particiones para trazabilidad ---
partitions_meta = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "train_count": len(train_dataset),
    "test_count": len(test_dataset),
    "val_count": len(val_dataset),
    "other_count": len(other_dataset) if other_dataset else 0,
    "splits_used": {
        "fine_tuning": "train",
        "benchmark_test": "test",
        "calibration": "validation (+ other)",
    },
}

partitions_path = DATA_DIR / "dataset_partitions.json"
with open(partitions_path, "w") as f:
    json.dump(partitions_meta, f, indent=2)
print(f"Metadatos de particiones guardados en: {partitions_path}")

Metadatos de particiones guardados en: /home/alfrog/projects/VibeVoice_Optimization/data/dataset_partitions.json


---
## TSK-1.3: Preprocesamiento Acustico

VibeVoice 1.5B espera audio crudo monofonico a **24,000 Hz** (no espectrogramas de Mel).
El pipeline de preprocesamiento consiste en:

1. **Remuestreo:** 48,000 Hz → 24,000 Hz (usando `librosa.resample`)
2. **Conversion a mono:** garantizado por el dataset
3. **Normalizacion de amplitud:** a -25 dB FS (mismo criterio que `AudioNormalizer` del modelo)
4. **Validacion:** descarte de audios corruptos o de duracion minima (<0.5s)

El resultado se almacena como tensores `torch.float32` en archivos `.pt`.

In [8]:
import librosa

TARGET_SR = 24000
TARGET_DB_FS = -25.0
MIN_DURATION_S = 0.5
EPS = 1e-6

def preprocess_audio(audio_array: np.ndarray, orig_sr: int) -> np.ndarray | None:
    """
    Preprocesa un array de audio para VibeVoice:
    remuestreo a 24kHz + normalizacion a target_dB_FS.
    Retorna None si el audio es invalido.
    """
    if audio_array is None or len(audio_array) == 0:
        return None

    # Remuestreo a 24kHz
    if orig_sr != TARGET_SR:
        audio = librosa.resample(
            y=audio_array.astype(np.float32),
            orig_sr=orig_sr,
            target_sr=TARGET_SR,
        )
    else:
        audio = audio_array.astype(np.float32)

    duration = len(audio) / TARGET_SR
    if duration < MIN_DURATION_S:
        return None

    # Normalizacion a target dB FS
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < EPS:
        return None
    scalar = (10 ** (TARGET_DB_FS / 20)) / (rms + EPS)
    audio = audio * scalar

    # Prevenir clipping
    peak = np.max(np.abs(audio))
    if peak > 1.0:
        audio = audio / peak

    return audio.astype(np.float32)

# --- Prueba con los primeros 5 registros ---
print("Validando preprocesamiento con 5 muestras...\n")
for i in range(5):
    sample = val_dataset[i]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    processed = preprocess_audio(arr, sr)
    if processed is not None:
        print(f"[{i}] OK: {len(processed)/TARGET_SR:.2f}s -> {processed.shape}, "
              f"peak={np.max(np.abs(processed)):.3f}, rms={np.sqrt(np.mean(processed**2)):.4f}")
    else:
        print(f"[{i}] DESCARTADO")

Validando preprocesamiento con 5 muestras...

[0] OK: 4.85s -> (116352,), peak=0.409, rms=0.0562
[1] OK: 5.57s -> (133632,), peak=0.453, rms=0.0562
[2] OK: 4.37s -> (104832,), peak=0.368, rms=0.0562
[3] OK: 5.18s -> (124416,), peak=0.358, rms=0.0562
[4] OK: 3.34s -> (80064,), peak=0.557, rms=0.0562


---
## TSK-1.4: Construccion del Set de Calibracion Offline

Se construye el tensor de calibracion a partir de la particion de validacion (10%).

**Estrategia:**
- Para GPTQ/AWQ se necesita un subconjunto representativo (tipicamente 128-512 muestras)
  que cubra diversidad de duraciones, generos y acentos.
- Se seleccionan `N_CALIB_SAMPLES` muestras con muestreo estratificado simple.
- Los audios preprocesados se concatenan en un unico tensor `[total_samples]`.
- Se guarda metadato con offsets para reconstruir limites de cada alocucion.

**Formato de salida:**
- `data/calibration_tensor.pt` — tensor float32 con audio concatenado
- `data/calibration_metadata.json` — indices originales, offsets, duraciones

In [9]:
N_CALIB_SAMPLES = 512
CALIB_SEED = 123

random.seed(CALIB_SEED)
np.random.seed(CALIB_SEED)

print(f"Seleccionando {N_CALIB_SAMPLES} muestras de calibracion...")

# --- Usar split validation como base para calibracion ---
calib_source = val_dataset
n_source = len(calib_source)
calib_local_indices = sorted(random.sample(range(n_source), min(N_CALIB_SAMPLES, n_source)))

# --- Preprocesar y recolectar ---
calib_audios = []
calib_metadata = []
rejected = 0
current_offset = 0

for local_idx in tqdm(calib_local_indices, desc="Preprocesando calibracion"):
    sample = calib_source[local_idx]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    processed = preprocess_audio(arr, sr)
    if processed is None:
        rejected += 1
        continue

    n_samples = len(processed)
    calib_audios.append(torch.from_numpy(processed))
    calib_metadata.append({
        "local_idx": local_idx,
        "sentence": sample["sentence"],
        "offset_start": current_offset,
        "offset_end": current_offset + n_samples,
        "n_samples": n_samples,
        "duration_s": n_samples / TARGET_SR,
        "gender": sample.get("gender", ""),
        "age": sample.get("age", ""),
    })
    current_offset += n_samples

print(f"\nMuestras procesadas : {len(calib_audios)}")
print(f"Muestras rechazadas : {rejected}")
print(f"Total de muestras de audio : {current_offset:,} ({current_offset/TARGET_SR:.1f}s)")

Seleccionando 512 muestras de calibracion...


Preprocesando calibracion: 100%|██████████| 512/512 [00:02<00:00, 190.64it/s]


Muestras procesadas : 512
Muestras rechazadas : 0
Total de muestras de audio : 74,889,543 (3120.4s)


In [10]:
# --- Concatenar en tensor unificado y guardar ---
calibration_tensor = torch.cat(calib_audios, dim=0)

tensor_path = DATA_DIR / "calibration_tensor.pt"
torch.save(calibration_tensor, tensor_path)

metadata_path = DATA_DIR / "calibration_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump({
        "dataset_id": DATASET_ID,
        "dataset_config": DATASET_CONFIG,
        "split_origin": "validation",
        "target_sr": TARGET_SR,
        "target_db_fs": TARGET_DB_FS,
        "n_samples": len(calib_audios),
        "total_audio_samples": calibration_tensor.shape[0],
        "total_duration_s": calibration_tensor.shape[0] / TARGET_SR,
        "records": calib_metadata,
    }, f, indent=2, ensure_ascii=False)

size_mb = tensor_path.stat().st_size / (1024 * 1024)
print(f"Tensor de calibracion : {tensor_path}")
print(f"Shape                 : {calibration_tensor.shape}")
print(f"Tamaño en disco       : {size_mb:.1f} MB")
print(f"Metadatos             : {metadata_path}")

Tensor de calibracion : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_tensor.pt
Shape                 : torch.Size([74889543])
Tamaño en disco       : 285.7 MB
Metadatos             : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_metadata.json


In [11]:
# --- Verificacion de carga ---
print("=== Verificacion del Set de Calibracion ===\n")

loaded_tensor = torch.load(DATA_DIR / "calibration_tensor.pt", weights_only=True)
with open(DATA_DIR / "calibration_metadata.json", "r") as f:
    loaded_meta = json.load(f)

print(f"Tensor cargado  : {loaded_tensor.shape}, dtype={loaded_tensor.dtype}")
print(f"Duracion total  : {loaded_meta['total_duration_s']:.1f}s")
print(f"Numero de clips : {loaded_meta['n_samples']}")
print(f"\nPrimeros 3 clips:")
for rec in loaded_meta["records"][:3]:
    print(f"  idx={rec['local_idx']:>6} | {rec['duration_s']:.2f}s | "
          f"\"{rec['sentence'][:60]}...\"")

# --- Verificar offsets ---
for i, rec in enumerate(loaded_meta["records"]):
    start, end = rec["offset_start"], rec["offset_end"]
    segment = loaded_tensor[start:end]
    assert len(segment) == rec["n_samples"], f"Offset mismatch en indice {i}"

print("\nVerificacion de offsets: OK")

=== Verificacion del Set de Calibracion ===

Tensor cargado  : torch.Size([74889543]), dtype=torch.float32
Duracion total  : 3120.4s
Numero de clips : 512

Primeros 3 clips:
  idx=     5 | 3.14s | "No lo he visto ...."
  idx=    27 | 5.14s | "Allá va el niño, donde le tratan con cariño...."
  idx=    97 | 6.82s | "de la mano , dando vueltas en setenta y cinco metros de eslo..."

Verificacion de offsets: OK


---
# Sprint 1.5: Fine-Tuning Monolingue Espanol con LoRA

**Objetivo:** Adaptar VibeVoice 1.5B a espanol exclusivamente mediante LoRA fine-tuning
sobre el split `train` de Common Voice 17.0. El modelo resultante (**VibeVoice-ES**)
sera la nueva linea base FP16 para todos los experimentos de cuantizacion.

**Estrategia:**
- LoRA rank=8 sobre proyecciones Q/K/V/O + gate/up/down del LLM (Qwen2.5)
- Entrenamiento completo del *diffusion head* para adaptar prosodia espanola
- Tokenizadores acustico y semantico congelados (lenguaje-agnosticos)
- `voice_prompt_drop_rate=1.0` para priorizar sintesis sobre clonacion
- `per_device_train_batch_size=1`, `gradient_accumulation_steps=64` para 8GB VRAM

> **Nota:** El entrenamiento ejecuta en terminal via `python -m vibevoice.finetune.train_vibevoice`.
> Las celdas de este bloque preparan los datos y generan el script de lanzamiento.
> La ejecucion del entrenamiento (~7-15h) se realiza fuera del notebook.

---
## TSK-1.5.1: Formatear Dataset al Esquema VibeVoice

El script de fine-tuning espera:
- Columna `text` con formato `"Speaker 1: {sentence}"`
- Columna `audio` como path a archivo de audio (str) o dict con array+sampling_rate
- Columna `voice_prompts` opcional (se auto-genera si no existe)

Guardamos los datos formateados como JSONL para pasarlos con `--train_jsonl` / `--validation_jsonl`.
Los paths de audio apuntan a la cache local de HuggingFace (archivos MP3 descargados).

In [12]:
import json
from pathlib import Path

FINETUNE_DATA_DIR = DATA_DIR / "finetune"
FINETUNE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Formateando train split a JSONL...")
train_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_train.jsonl"

with open(train_jsonl_path, "w", encoding="utf-8") as f:
    for i in tqdm(range(len(train_dataset)), desc="Escribiendo train JSONL"):
        sample = train_dataset[i]
        record = {
            "text": f"Speaker 1: {sample['sentence']}",
            "audio": sample["audio"]["path"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Train JSONL guardado: {train_jsonl_path}")
print(f"Tamano: {train_jsonl_path.stat().st_size / (1024**2):.1f} MB")

# --- Validation subset (2K del train para monitoreo) ---
print("\nCreando validation JSONL (muestra de 2000 registros)...")
val_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_val.jsonl"
N_VAL = min(2000, len(train_dataset))
val_indices_sample = sorted(random.sample(range(len(train_dataset)), N_VAL))

with open(val_jsonl_path, "w", encoding="utf-8") as f:
    for idx in tqdm(val_indices_sample, desc="Escribiendo val JSONL"):
        sample = train_dataset[idx]
        record = {
            "text": f"Speaker 1: {sample['sentence']}",
            "audio": sample["audio"]["path"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Validation JSONL guardado: {val_jsonl_path}")
print(f"Tamano: {val_jsonl_path.stat().st_size / (1024**2):.1f} MB")

Formateando train split a JSONL...


Escribiendo train JSONL: 100%|██████████| 336846/336846 [19:46<00:00, 283.80it/s] 


Train JSONL guardado: /home/alfrog/projects/VibeVoice_Optimization/data/finetune/cv17_es_train.jsonl
Tamano: 85.0 MB

Creando validation JSONL (muestra de 2000 registros)...


Escribiendo val JSONL: 100%|██████████| 2000/2000 [00:05<00:00, 335.11it/s]

Validation JSONL guardado: /home/alfrog/projects/VibeVoice_Optimization/data/finetune/cv17_es_val.jsonl
Tamano: 0.5 MB


---
## TSK-1.5.2: Script de Fine-Tuning para 8GB VRAM

Parametros optimizados para RTX 4060 Ti (8GB):

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `per_device_train_batch_size` | 1 | Limitar VRAM de activaciones |
| `gradient_accumulation_steps` | 64 | Batch efectivo = 64 |
| `bf16` | True | Mixed precision (Ada Lovelace) |
| `ddpm_batch_mul` | 1 | Minimo; sin amplificacion de difusion |
| `lora_r` | 8 | Rank estandar para adaptacion |
| `learning_rate` | 2.5e-5 | Recomendado por la comunidad |
| `num_train_epochs` | 1 | ~5,263 pasos (~7-15h) |
| `voice_prompt_drop_rate` | 1.0 | Drop siempre; prioriza sintesis |

El script generado se guarda en `scripts/run_finetune_es.sh`.

In [13]:
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

VENV_PYTHON = "/home/alfrog/micromamba/envs/vibevoice/bin/python"
MODEL_PATH = str(WEIGHTS_DIR / "vibevoice-1.5b")
OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / "finetune_vibevoice_es")
TRAIN_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_train.jsonl")
VAL_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_val.jsonl")

train_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Fine-Tuning VibeVoice 1.5B -> Espanol (LoRA)
# Hardware: RTX 4060 Ti 8GB VRAM
# Duracion estimada: 7-15 horas (1 epoch, 336K muestras)
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Iniciando Fine-Tuning VibeVoice-ES ==="
echo "Modelo base : {MODEL_PATH}"
echo "Train JSONL : {TRAIN_JSONL}"
echo "Val JSONL   : {VAL_JSONL}"
echo "Output      : {OUTPUT_DIR}"
echo ""

# Limpiar cache CUDA antes de empezar
{VENV_PYTHON} -c "import torch; torch.cuda.empty_cache(); print(f'VRAM libre: {{torch.cuda.memory_allocated()/1024**3:.2f}} GB')"

{VENV_PYTHON} -m vibevoice.finetune.train_vibevoice \\
    --model_name_or_path {MODEL_PATH} \\
    --train_jsonl {TRAIN_JSONL} \\
    --validation_jsonl {VAL_JSONL} \\
    --output_dir {OUTPUT_DIR} \\
    --text_column_name text \\
    --audio_column_name audio \\
    --per_device_train_batch_size 1 \\
    --gradient_accumulation_steps 64 \\
    --learning_rate 2.5e-5 \\
    --lr_scheduler_type cosine \\
    --warmup_ratio 0.03 \\
    --num_train_epochs 1 \\
    --logging_steps 50 \\
    --save_steps 500 \\
    --eval_steps 500 \\
    --save_total_limit 3 \\
    --bf16 True \\
    --do_train \\
    --do_eval \\
    --dataloader_num_workers 2 \\
    --remove_unused_columns False \\
    --gradient_checkpointing False \\
    --ddpm_batch_mul 1 \\
    --diffusion_loss_weight 1.4 \\
    --train_diffusion_head True \\
    --ce_loss_weight 0.04 \\
    --voice_prompt_drop_rate 1.0 \\
    --lora_r 8 \\
    --lora_alpha 32 \\
    --lora_dropout 0.05 \\
    --lora_target_modules q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj \\
    --max_grad_norm 0.8 \\
    --gradient_clipping \\
    --seed 42

echo ""
echo "=== Fine-Tuning completado ==="
echo "Checkpoint en: {OUTPUT_DIR}"
"""

script_path = SCRIPTS_DIR / "run_finetune_es.sh"
script_path.write_text(train_script)
script_path.chmod(0o755)

print(f"Script de entrenamiento generado: {script_path}")
print(f"\nPara ejecutar:\n  bash {script_path}")

Script de entrenamiento generado: /home/alfrog/projects/VibeVoice_Optimization/scripts/run_finetune_es.sh

Para ejecutar:
  bash /home/alfrog/projects/VibeVoice_Optimization/scripts/run_finetune_es.sh


---
## TSK-1.5.3: Merge de LoRA al Modelo Base

Al finalizar el fine-tuning, los adaptadores LoRA y el diffusion head entrenado
deben fusionarse al modelo base para obtener el checkpoint **VibeVoice-ES** en
`weights/vibevoice-1.5b-es/`. Se usa el script nativo `merge_vibevoice_models.py`.

In [14]:
MERGE_OUTPUT = str(WEIGHTS_DIR / "vibevoice-1.5b-es")

merge_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Merge LoRA -> VibeVoice-ES
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Mergeando LoRA + Diffusion Head al modelo base ==="
echo "Checkpoint LoRA : {OUTPUT_DIR}"
echo "Modelo base     : {MODEL_PATH}"
echo "Output (ES)     : {MERGE_OUTPUT}"
echo ""

{VENV_PYTHON} -m vibevoice.scripts.merge_vibevoice_models \\
    --base_model_path {MODEL_PATH} \\
    --checkpoint_path {OUTPUT_DIR}/lora \\
    --output_path {MERGE_OUTPUT} \\
    --output_format safetensors

echo ""
echo "=== VibeVoice-ES generado en: {MERGE_OUTPUT} ==="
ls -lh {MERGE_OUTPUT}/*.safetensors 2>/dev/null || echo "(verificar estructura)"
"""

merge_script_path = SCRIPTS_DIR / "merge_es_checkpoint.sh"
merge_script_path.write_text(merge_script)
merge_script_path.chmod(0o755)

print(f"Script de merge generado: {merge_script_path}")
print(f"\nPara ejecutar (despues del entrenamiento):\n  bash {merge_script_path}")

Script de merge generado: /home/alfrog/projects/VibeVoice_Optimization/scripts/merge_es_checkpoint.sh

Para ejecutar (despues del entrenamiento):
  bash /home/alfrog/projects/VibeVoice_Optimization/scripts/merge_es_checkpoint.sh


---
## TSK-1.5.4: Validacion del Modelo Fine-Tuneado

Verificamos que **VibeVoice-ES** carga correctamente en VRAM y registramos
el nuevo consumo base. La evaluacion completa de calidad se realiza en Sprint 2.

In [15]:
import sys, os
repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import torch
import transformers
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration

MODEL_ES_PATH = "../weights/vibevoice-1.5b-es"

print("Cargando VibeVoice-ES...")
torch.cuda.empty_cache()
model_es = VibeVoiceForConditionalGeneration.from_pretrained(
    MODEL_ES_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

vram_es = torch.cuda.memory_allocated() / 1024**3
print(f"VibeVoice-ES cargado. VRAM: {vram_es:.2f} GB")

# --- Verificacion de tamano en disco ---
import pathlib
model_size_mb = sum(f.stat().st_size for f in pathlib.Path(MODEL_ES_PATH).rglob("*.safetensors")) / (1024**2)
print(f"Tamano en disco (VibeVoice-ES): {model_size_mb:.1f} MB")

if vram_es < 6.0:
    print("VRAM OK - margen suficiente para cuantizacion.")
else:
    print("ADVERTENCIA: VRAM elevada. Verificar que el merge fue exitoso.")

APEX FusedRMSNorm not available, using native implementation


Cargando VibeVoice-ES...
Tied input and output embeddings using standard assignment.


Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.44s/it]

Tied input and output embeddings using standard assignment.
VibeVoice-ES cargado. VRAM: 5.04 GB
Tamano en disco (VibeVoice-ES): 10315.2 MB
VRAM OK - margen suficiente para cuantizacion.


---
## Sprint 1 - Resumen y Entregables

| Tarea | Estado | Artefacto |
|-------|--------|------------|
| TSK-1.1: Descarga | Completado | Dataset `fsicoli/common_voice_17_0` config `es` cargado |
| TSK-1.2: Particion | Completado | Splits nativos CV17: `train`/`test`/`validation` |
| TSK-1.3: Preprocesamiento | Completado | Funcion `preprocess_audio()` validada |
| TSK-1.4: Set de Calibracion | Completado | `data/calibration_tensor.pt` + `data/calibration_metadata.json` |

**Archivos generados en `data/`:**
- `dataset_partitions.json` — metadatos de los splits nativos CV17 (train/test/validation/other)
- `calibration_tensor.pt` — tensor float32 con audio de calibracion concatenado (512 muestras)
- `calibration_metadata.json` — offsets y metadatos de cada clip de calibracion

**Siguiente paso:** Ejecutar el script `scripts/run_finetune_es.sh` en terminal
para iniciar el fine-tuning (~7-15h). Luego mergear con `scripts/merge_es_checkpoint.sh`.